In [1]:
import os
import sys
import ctypes
from pathlib import Path
import numpy as np
import pandas as pd
import napari
import tifffile as tif
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Make NVRTC/NVIDIA DLLs discoverable before importing CuPy.
if os.name == 'nt':
    torch_lib = Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib'
    dll_dirs = [
        Path(sys.prefix) / 'Library' / 'bin',
        torch_lib,
    ]
    for dll_dir in dll_dirs:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

    if torch_lib.exists():
        os.environ['PATH'] = f"{torch_lib};" + os.environ.get('PATH', '')
        for name in ('nvrtc64_120_0.dll', 'nvrtc-builtins64_128.dll'):
            dll_path = torch_lib / name
            if dll_path.exists():
                ctypes.WinDLL(str(dll_path))

import cupy as cp
import cupyx.scipy.ndimage as ndi_gpu
import dask.array as da
from skimage.morphology import skeletonize as skeletonize_3d
import sknw
from scipy.spatial.distance import cdist
from scipy.ndimage import distance_transform_edt as edt
from scipy import ndimage as ndi

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def add_grouped_stats(df, group_masks, combined_name, exclude_cols):
    stats_out = {}
    for key, value in df.items():
        if key in exclude_cols:
            continue
        if not pd.api.types.is_numeric_dtype(value):
            continue

        arr = pd.to_numeric(value, errors='coerce').to_numpy(dtype=float)
        arr = arr[~np.isnan(arr)]
        stats_out[f'median_{combined_name}_{key}'] = float(np.median(arr)) if arr.size else np.nan

        for group_name, group_mask in group_masks.items():
            g_arr = pd.to_numeric(value[group_mask], errors='coerce').to_numpy(dtype=float)
            g_arr = g_arr[~np.isnan(g_arr)]
            stats_out[f'median_{group_name}_{key}'] = float(np.median(g_arr)) if g_arr.size else np.nan

    return stats_out

In [3]:
def cupy_chunk_processing(volume, processing_func, chunk_size=(64, 512, 512), overlap=(15, 15, 15), *args, **kwargs):
    result = np.empty_like(volume)
    pool = cp.get_default_memory_pool()
    z_steps = range(0, volume.shape[0], chunk_size[0])

    for z in tqdm(z_steps, desc='Processing chunks'):
        for y in range(0, volume.shape[1], chunk_size[1]):
            for x in range(0, volume.shape[2], chunk_size[2]):
                z_start = max(0, z - overlap[0])
                z_end = min(volume.shape[0], z + chunk_size[0] + overlap[0])
                y_start = max(0, y - overlap[1])
                y_end = min(volume.shape[1], y + chunk_size[1] + overlap[1])
                x_start = max(0, x - overlap[2])
                x_end = min(volume.shape[2], x + chunk_size[2] + overlap[2])

                chunk = volume[z_start:z_end, y_start:y_end, x_start:x_end]
                chunk_gpu = cp.asarray(chunk)

                def func_wrapper(gpu_chunk):
                    gpu_args = []
                    for arg in args:
                        gpu_args.append(cp.asarray(arg) if isinstance(arg, np.ndarray) else arg)

                    gpu_kwargs = {}
                    for k, v in kwargs.items():
                        gpu_kwargs[k] = cp.asarray(v) if isinstance(v, np.ndarray) else v

                    return processing_func(gpu_chunk, *gpu_args, **gpu_kwargs)

                filtered_chunk = func_wrapper(chunk_gpu)

                w_z_start = z - z_start
                w_z_end = w_z_start + chunk_size[0]
                w_y_start = y - y_start
                w_y_end = w_y_start + chunk_size[1]
                w_x_start = x - x_start
                w_x_end = w_x_start + chunk_size[2]

                valid_chunk = filtered_chunk[
                    w_z_start:min(w_z_end, filtered_chunk.shape[0]),
                    w_y_start:min(w_y_end, filtered_chunk.shape[1]),
                    w_x_start:min(w_x_end, filtered_chunk.shape[2]),
                ].get()

                result_z = min(z, result.shape[0])
                result_y = min(y, result.shape[1])
                result_x = min(x, result.shape[2])

                result[
                    result_z:result_z + valid_chunk.shape[0],
                    result_y:result_y + valid_chunk.shape[1],
                    result_x:result_x + valid_chunk.shape[2],
                ] = valid_chunk

                del chunk_gpu, filtered_chunk
                pool.free_all_blocks()

    return result

def measure_edge_length(coordinates):
    differences = np.diff(coordinates, axis=0)
    segment_lengths = np.linalg.norm(differences, axis=1)
    return np.sum(segment_lengths)

def prune_graph(graph, area_3d, edt_cutoff=0.25, length_cutoff=25):
    while True:
        endpoint_nodes = [node for node, degree in graph.degree() if degree == 1]
        values = []
        for node in endpoint_nodes:
            neighbors = list(graph.neighbors(node))

            if len(neighbors) == 1:
                neighbor = neighbors[0]
                edge_data = graph.get_edge_data(neighbor, node)
                edge_length = measure_edge_length(edge_data['pts'])
                branch_edt = area_3d[edge_data['pts'][:, 0], edge_data['pts'][:, 1], edge_data['pts'][:, 2]]
                branch_edt_interp = np.interp(np.linspace(0, 1, 100), np.linspace(0, 1, branch_edt.size), branch_edt)

                if np.mean(branch_edt_interp[:50]) < np.mean(branch_edt_interp[-50:]):
                    neighbor_coords = edge_data['pts'][-1]
                    part_oi = branch_edt_interp[:20]
                else:
                    neighbor_coords = edge_data['pts'][0]
                    part_oi = branch_edt_interp[-20:]

                neighbor_edt = area_3d[neighbor_coords[0], neighbor_coords[1], neighbor_coords[2]]
                value = np.mean(part_oi) / (neighbor_edt + 1e-6)

                if value > edt_cutoff or edge_length <= length_cutoff:
                    graph.remove_node(node)
                    values.append(value)

        if len(values) == 0:
            break
    return graph

def remove_mid_node(graph):
    while True:
        nodes_to_process = [n for n, d in graph.degree() if d == 2]
        if not nodes_to_process:
            break

        processed_in_iteration = False
        for i in nodes_to_process:
            if not graph.has_node(i) or graph.degree(i) != 2:
                continue

            neighbors = list(graph.neighbors(i))
            if len(neighbors) != 2:
                continue

            n1, n2 = neighbors[0], neighbors[1]
            if n1 == n2 or graph.has_edge(n1, n2):
                continue

            edge1 = graph.get_edge_data(i, n1)
            edge2 = graph.get_edge_data(i, n2)
            pts1 = np.atleast_2d(edge1['pts'])
            pts2 = np.atleast_2d(edge2['pts'])
            node_coord = graph.nodes[i]['pts'].astype(np.int32)

            s1, e1 = pts1[0], pts1[-1]
            s2, e2 = pts2[0], pts2[-1]

            dists = cdist([s1, e1], [s2, e2])
            min_row, min_col = np.unravel_index(np.argmin(dists), dists.shape)

            if min_row == 0 and min_col == 0:
                combined_line = np.concatenate([pts1[::-1], [node_coord], pts2], axis=0)
            elif min_row == 1 and min_col == 1:
                combined_line = np.concatenate([pts1, [node_coord], pts2[::-1]], axis=0)
            elif min_row == 0 and min_col == 1:
                combined_line = np.concatenate([pts2[::-1], [node_coord], pts1], axis=0)
            else:
                combined_line = np.concatenate([pts1, [node_coord], pts2], axis=0)

            new_weight = edge1.get('weight', 0) + edge2.get('weight', 0)
            graph.add_edge(n1, n2, weight=new_weight, pts=combined_line)
            graph.remove_node(i)
            processed_in_iteration = True

        if not processed_in_iteration:
            break
    return graph

def collect_border_vicinity_edges(graph, image_shape, vicinity_z=1, vicinity_xy=50):
    border_vicinity_edges = set()
    for u, v in graph.edges():
        try:
            pts = graph[u][v]['pts']
            if any((
                # pt[0] < vicinity_z or pt[0] > image_shape[0] - 1 - vicinity_z or
                pt[1] < vicinity_xy or pt[1] > image_shape[1] - 1 - vicinity_xy or
                pt[2] < vicinity_xy or pt[2] > image_shape[2] - 1 - vicinity_xy
                ) for pt in pts):
                border_vicinity_edges.add((u, v))
        except KeyError:
            continue

    trimmed_subgraph = graph.copy()
    edges_to_remove = [edge for edge in border_vicinity_edges if trimmed_subgraph.has_edge(*edge)]
    trimmed_subgraph.remove_edges_from(edges_to_remove)

    isolated_nodes = [node for node in trimmed_subgraph.nodes() if trimmed_subgraph.degree[node] == 0]
    if isolated_nodes:
        trimmed_subgraph.remove_nodes_from(isolated_nodes)

    return trimmed_subgraph

def compute_cross_sectional_areas(mask, skeleton, binary_edt, voxel_size_um=(2.0, 2.0, 2.0)):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    edt_2d = edt(np.max(mask, axis=0), sampling=tuple(voxel_size_um[1:]))
    area_3d = np.zeros_like(binary_edt, dtype=float)
    z_idx, y_idx, x_idx = np.where(skeleton > 0)

    minor_axis = 2 * binary_edt[z_idx, y_idx, x_idx]
    major_axis = 2 * edt_2d[y_idx, x_idx]

    areas = np.pi * (major_axis / 2) * (minor_axis / 2)
    area_3d[z_idx, y_idx, x_idx] = areas
    return area_3d

def compute_vessel_metrics(graph, area_image, vessel_metrics_df, voxel_size_um=(2.0, 2.0, 2.0)):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    zs, ys, xs = [], [], []
    volumes, lengths, shortest_paths = [], [], []
    tortuosities, is_sprouts = [], []
    mean_cs_areas, median_cs_areas, std_cs_areas = [], [], []
    node1_degrees, node2_degrees = [], []
    orientation_zs, orientation_ys, orientation_xs = [], [], []

    edges = list(graph.edges())
    valid_edges = []
    if not edges:
        cols = [
            'z', 'y', 'x', 'volume', 'length', 'shortest_path', 'tortuosity',
            'is_sprout', 'mean_cs_area', 'median_cs_area', 'std_cs_area',
            'node1_degree', 'node2_degree', 'orientation_z', 'orientation_y', 'orientation_x'
        ]
        return pd.DataFrame(columns=cols)

    node_degrees_dict = dict(graph.degree())

    for u, v in edges:
        try:
            pts = graph[u][v]['pts']
            if len(pts) < 2:
                continue

            pts_um = np.asarray(pts, dtype=float) * voxel_size_um[None, :]
            segment_lengths_um = np.linalg.norm(np.diff(pts_um, axis=0), axis=1)

            valid_edges.append((u, v))
            zs.append(pts[:, 0])
            ys.append(pts[:, 1])
            xs.append(pts[:, 2])

            segment_areas = area_image[pts[:, 0], pts[:, 1], pts[:, 2]]
            mean_cs_areas.append(np.nanmean(segment_areas))
            median_cs_areas.append(np.nanmedian(segment_areas))
            std_cs_areas.append(np.nanstd(segment_areas))

            if segment_areas.size > 1 and segment_lengths_um.size > 0:
                edge_volume = np.nansum(0.5 * (segment_areas[:-1] + segment_areas[1:]) * segment_lengths_um)
            elif segment_areas.size == 1:
                edge_volume = float(segment_areas[0])
            else:
                edge_volume = np.nan
            volumes.append(float(edge_volume))

            l = float(np.sum(segment_lengths_um))
            lengths.append(l)
            sp = float(np.linalg.norm(pts_um[0] - pts_um[-1]))
            shortest_paths.append(sp)
            tortuosities.append(np.clip(l / (sp + 1e-8), 0, 5))

            deg_u = node_degrees_dict.get(u, 0)
            deg_v = node_degrees_dict.get(v, 0)
            node1_degrees.append(deg_u)
            node2_degrees.append(deg_v)

            is_sprouts.append(graph.nodes[u]['sprout'] or graph.nodes[v]['sprout'])

            direction_vector = pts_um[-1] - pts_um[0]
            norm = np.linalg.norm(direction_vector)
            if norm > 1e-8:
                normalized_vector = direction_vector / norm
            else:
                normalized_vector = np.array([np.nan, np.nan, np.nan])

            orientation_zs.append(normalized_vector[0])
            orientation_ys.append(normalized_vector[1])
            orientation_xs.append(normalized_vector[2])
        except (KeyError, IndexError):
            continue

    vessel_metrics_df = pd.DataFrame(index=pd.MultiIndex.from_tuples(valid_edges, names=['node1', 'node2']))

    vessel_metrics_df['z'] = zs
    vessel_metrics_df['y'] = ys
    vessel_metrics_df['x'] = xs
    vessel_metrics_df['volume'] = volumes
    vessel_metrics_df['length'] = lengths
    vessel_metrics_df['shortest_path'] = shortest_paths
    vessel_metrics_df['tortuosity'] = tortuosities
    vessel_metrics_df['is_sprout'] = is_sprouts
    vessel_metrics_df['mean_cs_area'] = mean_cs_areas
    vessel_metrics_df['median_cs_area'] = median_cs_areas
    vessel_metrics_df['std_cs_area'] = std_cs_areas
    vessel_metrics_df['node1_degree'] = node1_degrees
    vessel_metrics_df['node2_degree'] = node2_degrees
    vessel_metrics_df['orientation_z'] = orientation_zs
    vessel_metrics_df['orientation_y'] = orientation_ys
    vessel_metrics_df['orientation_x'] = orientation_xs

    return vessel_metrics_df

def compute_junction_metrics(graph, junction_metrics_df, distance_threshold=1000.0, voxel_size_um=(2.0, 2.0, 2.0)):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    cols = ['z', 'y', 'x', 'number_of_vessel_per_node', 'node_type', 'dist_nearest_junction', 'dist_nearest_endpoint', 'num_junction_neighbors', 'num_endpoint_neighbors']
    nodes = list(graph.nodes())
    if not nodes:
        return pd.DataFrame(index=nodes, columns=cols)

    positions = np.array([graph.nodes[n]['pts'] for n in nodes])
    positions_um = positions.astype(float) * voxel_size_um[None, :]
    node_type = np.array(['sprout' if graph.nodes[n]['sprout'] else 'junction' for n in nodes])
    junction_metrics_df = pd.DataFrame(index=nodes)
    junction_metrics_df['z'] = positions_um[:, 0]
    junction_metrics_df['y'] = positions_um[:, 1]
    junction_metrics_df['x'] = positions_um[:, 2]
    junction_metrics_df['number_of_vessel_per_node'] = np.array([graph.degree[n] for n in nodes])
    junction_metrics_df['node_type'] = node_type
    junction_metrics_df['dist_nearest_junction'] = np.nan
    junction_metrics_df['dist_nearest_endpoint'] = np.nan
    junction_metrics_df['num_junction_neighbors'] = 0
    junction_metrics_df['num_endpoint_neighbors'] = 0

    if len(nodes) < 2:
        return junction_metrics_df

    dist_matrix = cdist(positions_um, positions_um)
    endpoint_mask = (node_type == 'sprout')
    branch_point_mask = (node_type == 'junction')

    def fill_group_metrics(target_mask, reference_mask, dist_col, count_col, same_group=False):
        if not np.any(target_mask) or not np.any(reference_mask):
            return
        sub_dist = dist_matrix[np.ix_(target_mask, reference_mask)].copy()
        if same_group:
            np.fill_diagonal(sub_dist, np.inf)
        junction_metrics_df.loc[target_mask, dist_col] = np.min(sub_dist, axis=1)
        junction_metrics_df.loc[target_mask, count_col] = np.sum(sub_dist <= distance_threshold, axis=1)

    fill_group_metrics(branch_point_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors', same_group=True)
    fill_group_metrics(branch_point_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors')
    fill_group_metrics(endpoint_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors')
    fill_group_metrics(endpoint_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors', same_group=True)
    return junction_metrics_df

def fractal_dimension_and_lacunarity(binary, min_box_size=1, max_box_size=None, n_samples=12):
    pts = np.argwhere(binary > 0)
    if pts.size == 0:
        return np.nan, np.nan

    if max_box_size is None:
        max_box_size = int(np.floor(np.log2(np.min(binary.shape))))

    scales = np.floor(np.logspace(max_box_size, min_box_size, num=n_samples, base=2)).astype(np.int64)
    scales = np.unique(scales)
    scales = scales[scales > 0]

    log_inv_scale = []
    log_N = []
    lac_vals = []

    for s in scales:
        box_ids = pts // s
        unique_box_ids, counts = np.unique(box_ids, axis=0, return_counts=True)  # occupied boxes + masses
        N = unique_box_ids.shape[0]
        if N < 2:
            continue

        log_inv_scale.append(np.log(1.0 / s))
        log_N.append(np.log(N))

        mu = counts.mean()
        lac_vals.append((counts.var() / (mu * mu)) if mu > 0 else np.nan)

    if len(log_N) < 2:
        return np.nan, np.nan

    fd = float(np.polyfit(log_inv_scale, log_N, 1)[0])
    lac = float(np.nanmean(lac_vals)) if len(lac_vals) else np.nan
    return fd, lac


def graph2image(graph, shape):
    pruned_skeleton = np.zeros(shape)
    for u, v in graph.edges():
        coords = graph.get_edge_data(u, v)['pts']

        clipped_coords = np.zeros_like(coords)
        clipped_coords[:, 0] = np.clip(coords[:, 0], 0, shape[0] - 1)
        clipped_coords[:, 1] = np.clip(coords[:, 1], 0, shape[1] - 1)
        clipped_coords[:, 2] = np.clip(coords[:, 2], 0, shape[2] - 1)

        pruned_skeleton[clipped_coords[:, 0], clipped_coords[:, 1], clipped_coords[:, 2]] = 1

    return pruned_skeleton

In [16]:
import time
import networkx as nx

def compute_junction_metrics(
    graph,
    junction_metrics_df,
    distance_threshold=1000.0,
    voxel_size_um=(2.0, 2.0, 2.0),
    distance_mode='skeleton',
    report_timing=True,
    ):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    cols = [
        'z', 'y', 'x',
        'number_of_vessel_per_node', 'node_type',
        'dist_nearest_junction', 'dist_nearest_endpoint',
        'num_junction_neighbors', 'num_endpoint_neighbors'
    ]
    nodes = list(graph.nodes())
    if not nodes:
        return pd.DataFrame(index=nodes, columns=cols)

    positions = np.array([graph.nodes[n]['pts'] for n in nodes])
    positions_um = positions.astype(float) * voxel_size_um[None, :]
    node_type = np.array(['sprout' if graph.nodes[n]['sprout'] else 'junction' for n in nodes])

    junction_metrics_df = pd.DataFrame(index=nodes)
    junction_metrics_df['z'] = positions_um[:, 0]
    junction_metrics_df['y'] = positions_um[:, 1]
    junction_metrics_df['x'] = positions_um[:, 2]
    junction_metrics_df['number_of_vessel_per_node'] = np.array([graph.degree[n] for n in nodes])
    junction_metrics_df['node_type'] = node_type
    junction_metrics_df['dist_nearest_junction'] = np.nan
    junction_metrics_df['dist_nearest_endpoint'] = np.nan
    junction_metrics_df['num_junction_neighbors'] = 0
    junction_metrics_df['num_endpoint_neighbors'] = 0

    if len(nodes) < 2:
        return junction_metrics_df

    t0 = time.perf_counter()

    if distance_mode == 'skeleton':
        graph_weighted = graph.copy()
        for u, v, data in graph_weighted.edges(data=True):
            pts = data.get('pts', None)
            if pts is None or len(pts) < 2:
                graph_weighted.edges[u, v]['path_length_um'] = np.inf
                continue
            pts_um = np.asarray(pts, dtype=float) * voxel_size_um[None, :]
            seg_lengths = np.linalg.norm(np.diff(pts_um, axis=0), axis=1)
            graph_weighted.edges[u, v]['path_length_um'] = float(np.sum(seg_lengths))

        node_to_idx = {n: i for i, n in enumerate(nodes)}
        dist_matrix = np.full((len(nodes), len(nodes)), np.inf, dtype=float)
        for source in nodes:
            i = node_to_idx[source]
            lengths = nx.single_source_dijkstra_path_length(
                graph_weighted, source, weight='path_length_um'
            )
            for target, d in lengths.items():
                j = node_to_idx[target]
                dist_matrix[i, j] = float(d)
    else:
        dist_matrix = cdist(positions_um, positions_um)

    elapsed_s = time.perf_counter() - t0
    if report_timing:
        print(f"compute_junction_metrics distance_mode={distance_mode} took {elapsed_s:.3f} s for {len(nodes)} nodes")

    endpoint_mask = (node_type == 'sprout')
    branch_point_mask = (node_type == 'junction')

    def fill_group_metrics(target_mask, reference_mask, dist_col, count_col, same_group=False):
        if not np.any(target_mask) or not np.any(reference_mask):
            return
        sub_dist = dist_matrix[np.ix_(target_mask, reference_mask)].copy()
        if same_group:
            np.fill_diagonal(sub_dist, np.inf)

        nearest = np.min(sub_dist, axis=1)
        nearest[~np.isfinite(nearest)] = np.nan
        junction_metrics_df.loc[target_mask, dist_col] = nearest
        junction_metrics_df.loc[target_mask, count_col] = np.sum(sub_dist <= distance_threshold, axis=1)

    fill_group_metrics(branch_point_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors', same_group=True)
    fill_group_metrics(branch_point_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors')
    fill_group_metrics(endpoint_mask, branch_point_mask, 'dist_nearest_junction', 'num_junction_neighbors')
    fill_group_metrics(endpoint_mask, endpoint_mask, 'dist_nearest_endpoint', 'num_endpoint_neighbors', same_group=True)

    return junction_metrics_df

In [4]:
vasculature_segmentation= np.load(r"C:\Users\taylorhearn\git_repos\vascumap\working_outputs_double_crop\20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10_vessel_mask_iso_354_cropped.npy")

In [19]:
##################### CLEAN SEGMENTATION #####################
voxel_size_um = np.array([2.0, 2.0, 2.0], dtype=float)
voxel_volume_um3 = float(np.prod(voxel_size_um))
chunk_size = (vasculature_segmentation.shape[0], 512, 512)
binary_smoothed = cupy_chunk_processing(volume=vasculature_segmentation.astype(np.float32), processing_func=ndi_gpu.gaussian_filter, sigma=3, chunk_size=chunk_size) > 0.5
binary_filled_holes = cupy_chunk_processing(volume=binary_smoothed, processing_func=ndi_gpu.binary_fill_holes, chunk_size=chunk_size).astype(np.float32)
binary_edt = cupy_chunk_processing(volume=binary_filled_holes, processing_func=ndi_gpu.distance_transform_edt, sampling=tuple(voxel_size_um), chunk_size=chunk_size)

##################### SKELETONISE AND GRAPH #####################
dask_volume = da.from_array(binary_filled_holes.astype(bool), chunks=chunk_size)
skeleton = da.overlap.map_overlap(dask_volume, skeletonize_3d, depth=(2, 2, 2), boundary='none', dtype=bool).compute(scheduler='threads')
graph = sknw.build_sknw(skeleton)
area_image = compute_cross_sectional_areas(vasculature_segmentation, skeleton, binary_edt, voxel_size_um=voxel_size_um) # colour = area of vasculature at that point on skeleton

for n in list(graph.nodes()):
    graph.nodes[n]['pts'] = graph.nodes[n]['pts'][0]
    graph.nodes[n]['sprout'] = graph.degree(n) == 1

pruned_graph = prune_graph(graph=graph, area_3d=area_image, edt_cutoff=0.20, length_cutoff=25)
clean_graph = remove_mid_node(pruned_graph)
clean_graph = collect_border_vicinity_edges(clean_graph, vasculature_segmentation.shape)
isolated_nodes = [node for node in clean_graph.nodes() if clean_graph.degree[node] == 0]
if isolated_nodes:
    clean_graph.remove_nodes_from(isolated_nodes)

# Convert final cleaned graph back to a 3D skeleton volume
skeleton_from_graph = graph2image(clean_graph, vasculature_segmentation.shape).astype(np.uint8)
for node in clean_graph.nodes():
    clean_graph.nodes[node]['sprout'] = clean_graph.degree(node) == 1

##################### METRICS #####################
junction_metrics_df = pd.DataFrame()
vessel_metrics_df = pd.DataFrame()
global_metrics = {}

chip_volume_um3 = float(np.prod(vasculature_segmentation.shape)) * voxel_volume_um3
vessel_volume_um3 = float(np.count_nonzero(binary_filled_holes)) * voxel_volume_um3
neighbor_distance_threshold_um = 250.0 * float(voxel_size_um[1])
junction_distance_mode = 'skeleton'  # switch to 'euclidean' for faster runtime

def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator <= 0:
        return np.nan
    return float(numerator) / denominator

def safe_median(values):
    arr = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.median(arr))

def compute_internal_pore_headline_metrics(
    mask,
    voxel_size_um=(2.0, 2.0, 2.0),
    min_pore_area_um2=16.0,
    max_pore_area_fraction_of_slice=0.15,
    use_gpu_edt=True,
    ):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    _, y_um, x_um = voxel_size_um
    pixel_area_um2 = float(y_um * x_um)

    total_pore_area_um2 = 0.0
    total_filled_area_um2 = 0.0
    pore_areas_all = []
    pore_radii_all = []

    for z in range(mask.shape[0]):
        vessel_slice = mask[z].astype(bool)
        filled_slice = ndi.binary_fill_holes(vessel_slice)
        internal_pores = filled_slice & ~vessel_slice

        filled_area_um2 = float(np.count_nonzero(filled_slice)) * pixel_area_um2
        total_filled_area_um2 += filled_area_um2

        if not np.any(internal_pores):
            continue

        labeled, n_labels = ndi.label(internal_pores, structure=np.ones((3, 3), dtype=np.uint8))
        if n_labels == 0:
            continue

        area_counts = np.bincount(labeled.ravel(), minlength=n_labels + 1)[1:].astype(np.float64)
        area_um2_all = area_counts * pixel_area_um2

        slice_area_um2 = float(vessel_slice.size) * pixel_area_um2
        max_pore_area_um2 = float(max_pore_area_fraction_of_slice) * slice_area_um2

        label_ids = np.arange(1, n_labels + 1, dtype=np.int32)
        valid_mask = (area_um2_all >= min_pore_area_um2) & (area_um2_all <= max_pore_area_um2)
        if not np.any(valid_mask):
            continue

        valid_label_ids = label_ids[valid_mask]
        valid_areas = area_um2_all[valid_mask]

        if use_gpu_edt:
            dist_map_um = cp.asnumpy(
                ndi_gpu.distance_transform_edt(
                    cp.asarray(internal_pores, dtype=cp.uint8),
                    sampling=(float(y_um), float(x_um)),
                )
            )
        else:
            dist_map_um = edt(internal_pores, sampling=(y_um, x_um))

        valid_radii = np.asarray(
            ndi.maximum(dist_map_um, labels=labeled, index=valid_label_ids),
            dtype=float,
        )

        pore_areas_all.append(valid_areas)
        pore_radii_all.append(valid_radii)
        total_pore_area_um2 += float(np.sum(valid_areas))

    if len(pore_areas_all) == 0:
        return {
            'total_internal_pore_count': 0,
            'internal_pore_area_fraction_in_filled_vascular_area': 0.0,
            'median_internal_pore_area_um2': np.nan,
            'p90_internal_pore_area_um2': np.nan,
            'median_internal_pore_max_inscribed_radius_um': np.nan,
        }

    all_areas = np.concatenate(pore_areas_all)
    all_radii = np.concatenate(pore_radii_all)

    return {
        'total_internal_pore_count': int(all_areas.size),
        'internal_pore_area_fraction_in_filled_vascular_area': float(total_pore_area_um2 / max(total_filled_area_um2, 1e-12)),
        'median_internal_pore_area_um2': float(np.median(all_areas)),
        'p90_internal_pore_area_um2': float(np.percentile(all_areas, 90)),
        'median_internal_pore_max_inscribed_radius_um': float(np.median(all_radii)),
    }

if clean_graph.number_of_edges() > 0:
    fd, lacunarity = fractal_dimension_and_lacunarity(area_image > 0)
    total_vessel_length_um = np.sum([
        np.linalg.norm(
            np.diff(clean_graph[u][v]['pts'].astype(float) * voxel_size_um[None, :], axis=0),
            axis=1
        ).sum()
        for u, v in clean_graph.edges()
    ])
    branchpoints_count = sum(1 for u in clean_graph.nodes() if not clean_graph.nodes[u]['sprout'])
    sprouts_count = sum(1 for u, v in clean_graph.edges() if clean_graph.nodes[u]['sprout'] or clean_graph.nodes[v]['sprout'])
else:
    fd, lacunarity = np.nan, np.nan
    total_vessel_length_um = 0.0
    branchpoints_count = 0
    sprouts_count = 0

global_metrics['chip_volume_um3'] = chip_volume_um3
global_metrics['vessel_volume_um3'] = vessel_volume_um3
global_metrics['vessel_volume_fraction'] = safe_divide(vessel_volume_um3, chip_volume_um3)
global_metrics['total_vessel_length_um'] = float(total_vessel_length_um)
global_metrics['vessel_length_per_chip_volume_um_inverse2'] = safe_divide(total_vessel_length_um, chip_volume_um3)
global_metrics['sprouts_per_vessel_length_um_inverse'] = safe_divide(sprouts_count, total_vessel_length_um)
global_metrics['junctions_per_vessel_length_um_inverse'] = safe_divide(branchpoints_count, total_vessel_length_um)
global_metrics['fractal_dimension'] = fd
global_metrics['lacunarity'] = lacunarity
global_metrics['median_sprout_and_branch_tortuosity'] = np.nan
global_metrics['median_sprout_and_branch_median_cs_area_um2'] = np.nan
global_metrics['median_junction_dist_nearest_junction_um'] = np.nan
global_metrics['median_sprout_dist_nearest_endpoint_um'] = np.nan

if clean_graph.number_of_nodes() > 0 and clean_graph.number_of_edges() > 0:
    vessel_metrics_df = compute_vessel_metrics(clean_graph, area_image, vessel_metrics_df, voxel_size_um=voxel_size_um)
    junction_metrics_df = compute_junction_metrics(
        clean_graph,
        junction_metrics_df,
        distance_threshold=neighbor_distance_threshold_um,
        voxel_size_um=voxel_size_um,
        distance_mode=junction_distance_mode,
    )

    if not vessel_metrics_df.empty:
        global_metrics['median_sprout_and_branch_tortuosity'] = safe_median(vessel_metrics_df['tortuosity'])
        global_metrics['median_sprout_and_branch_median_cs_area_um2'] = safe_median(vessel_metrics_df['median_cs_area'])

    if not junction_metrics_df.empty:
        is_branch_point_mask = (junction_metrics_df['node_type'] == 'junction')
        is_sprout_tip_mask = (junction_metrics_df['node_type'] == 'sprout')

        if np.any(is_branch_point_mask):
            global_metrics['median_junction_dist_nearest_junction_um'] = safe_median(
                junction_metrics_df.loc[is_branch_point_mask, 'dist_nearest_junction']
            )

        if np.any(is_sprout_tip_mask):
            global_metrics['median_sprout_dist_nearest_endpoint_um'] = safe_median(
                junction_metrics_df.loc[is_sprout_tip_mask, 'dist_nearest_endpoint']
            )

pore_global_metrics = compute_internal_pore_headline_metrics(
    binary_filled_holes.astype(bool),
    voxel_size_um=voxel_size_um,
    min_pore_area_um2=16.0,
    max_pore_area_fraction_of_slice=0.15,
    use_gpu_edt=True,
    )
global_metrics.update(pore_global_metrics)

global_metrics_df = pd.DataFrame([global_metrics])

Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.96s/it]
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\dask\array\overlap.py:667: FutureWarning: The use of map_overlap(array, func, **kwargs) is deprecated since dask 2.17.0 and will be an error in a future release. To silence this warning, use the syntax map_overlap(func, array0,[ array1, ...,] **kwargs) instead.
  warnings.warn(


compute_junction_metrics distance_mode=skeleton took 1.001 s for 2036 nodes


In [18]:
# Quick runtime comparison: skeleton-path vs euclidean node distances
_ = compute_junction_metrics(
    clean_graph,
    pd.DataFrame(),
    distance_threshold=neighbor_distance_threshold_um,
    voxel_size_um=voxel_size_um,
    distance_mode='skeleton',
    report_timing=True,
)
_ = compute_junction_metrics(
    clean_graph,
    pd.DataFrame(),
    distance_threshold=neighbor_distance_threshold_um,
    voxel_size_um=voxel_size_um,
    distance_mode='euclidean',
    report_timing=True,
)

compute_junction_metrics distance_mode=skeleton took 1.066 s for 2036 nodes
compute_junction_metrics distance_mode=euclidean took 0.011 s for 2036 nodes


In [6]:
# ---------------- Shape-invariant normalization layer ----------------
chip_extent_um = np.asarray(vasculature_segmentation.shape, dtype=float) * voxel_size_um
chip_characteristic_length_um = float(np.cbrt(max(chip_volume_um3, 1e-12)))
chip_characteristic_area_um2 = chip_characteristic_length_um ** 2

global_metrics['chip_characteristic_length_um'] = chip_characteristic_length_um

# Core extensive metrics -> one normalized variant per concept
global_metrics['total_vessel_length_per_chip_characteristic_length'] = safe_divide(
    global_metrics.get('total_vessel_length_um', np.nan), chip_characteristic_length_um
)

global_metrics['sprouts_per_chip_volume_um_inverse3'] = safe_divide(
    sprouts_count, chip_volume_um3
)
global_metrics['junctions_per_chip_volume_um_inverse3'] = safe_divide(
    branchpoints_count, chip_volume_um3
)

# Distance-like metrics -> normalized by characteristic chip length
global_metrics['median_junction_dist_nearest_junction_per_characteristic_length'] = safe_divide(
    global_metrics.get('median_junction_dist_nearest_junction_um', np.nan), chip_characteristic_length_um
)
global_metrics['median_sprout_dist_nearest_endpoint_per_characteristic_length'] = safe_divide(
    global_metrics.get('median_sprout_dist_nearest_endpoint_um', np.nan), chip_characteristic_length_um
)

# Area-like metric -> normalized by characteristic area
global_metrics['median_cs_area_over_characteristic_area'] = safe_divide(
    global_metrics.get('median_sprout_and_branch_median_cs_area_um2', np.nan), chip_characteristic_area_um2
)

# Pore count normalization -> vessel-volume normalized only
global_metrics['total_internal_pore_density_per_vessel_volume_um_inverse3'] = safe_divide(
    global_metrics.get('total_internal_pore_count', np.nan), vessel_volume_um3
)

# Rebuild DF with normalized metrics included
global_metrics_df = pd.DataFrame([global_metrics])

In [13]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation", opacity=0.2)
viewer.add_labels(binary_filled_holes.astype(np.uint8), name="filled")

# Per-slice internal holes: filled_slice & ~vessel_slice, filtered by max area fraction
max_hole_area_fraction_of_slice = 0.10
holes = np.zeros_like(vasculature_segmentation, dtype=np.uint8)
hole_labels_per_slice = np.zeros_like(vasculature_segmentation, dtype=np.int32)
hole_distance_per_slice_um = np.zeros_like(vasculature_segmentation, dtype=np.float32)

for z in range(vasculature_segmentation.shape[0]):
    vessel_slice = vasculature_segmentation[z].astype(bool)
    filled_slice = ndi.binary_fill_holes(vessel_slice)
    internal_holes_slice = filled_slice & ~vessel_slice

    labeled_slice, n_labels = ndi.label(internal_holes_slice, structure=np.ones((3, 3), dtype=np.uint8))
    if n_labels == 0:
        continue

    max_hole_area_px = float(max_hole_area_fraction_of_slice) * float(vessel_slice.size)
    area_counts = np.bincount(labeled_slice.ravel(), minlength=n_labels + 1).astype(np.float64)
    valid_label_mask = (area_counts > 0) & (area_counts <= max_hole_area_px)
    valid_label_mask[0] = False

    filtered_holes_slice = valid_label_mask[labeled_slice]
    holes[z] = filtered_holes_slice.astype(np.uint8)

    relabeled_slice, _ = ndi.label(filtered_holes_slice, structure=np.ones((3, 3), dtype=np.uint8))
    hole_labels_per_slice[z] = relabeled_slice.astype(np.int32)

    if np.any(filtered_holes_slice):
        hole_distance_per_slice_um[z] = edt(
            filtered_holes_slice,
            sampling=tuple(voxel_size_um[1:]),
        ).astype(np.float32)

viewer.add_labels(holes, name="holes")
viewer.add_labels(hole_labels_per_slice, name="hole_labels_per_slice")
viewer.add_image(hole_distance_per_slice_um, name="hole_distance_per_slice_um", colormap="magma", opacity=0.6)
viewer.add_labels(skeleton.astype(np.uint8), name="raw_skeleton")
viewer.add_labels(skeleton_from_graph, name="pruned_graph_skeleton")

# Optional: show graph nodes as points
node_pts = np.array([clean_graph.nodes[n]["pts"] for n in clean_graph.nodes()], dtype=float)
if len(node_pts) > 0:
    viewer.add_points(node_pts, name="graph_nodes")

In [8]:
global_metrics_df

,chip_volume_um3,vessel_volume_um3,vessel_volume_fraction,total_vessel_length_um,vessel_length_per_chip_volume_um_inverse2,sprouts_per_vessel_length_um_inverse,junctions_per_vessel_length_um_inverse,fractal_dimension,lacunarity,median_sprout_and_branch_tortuosity,...,p90_internal_pore_area_um2,median_internal_pore_max_inscribed_radius_um,chip_characteristic_length_um,total_vessel_length_per_chip_characteristic_length,sprouts_per_chip_volume_um_inverse3,junctions_per_chip_volume_um_inverse3,median_junction_dist_nearest_junction_per_characteristic_length,median_sprout_dist_nearest_endpoint_per_characteristic_length,median_cs_area_over_characteristic_area,total_internal_pore_density_per_vessel_volume_um_inverse3
0,1.439690e+09,632134432.0,0.439077,303629.658248,0.000211,0.00084,0.005859,1.129302,0.324819,1.211726,...,18602.4,18.439089,1129.162239,268.898169,1.771214e-07,0.000001,0.025606,0.086573,0.002599,0.000023


In [41]:
# Holes are background voxels (not bitwise inversion of integer labels)
holes = (vasculature_segmentation == 0)

# Compute EDT of holes with the same chunked GPU pipeline
hole_distance = cupy_chunk_processing(
    volume=holes.astype(np.float32),
    processing_func=ndi_gpu.distance_transform_edt,
    chunk_size=chunk_size
    )

Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.61s/it]


In [42]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation", opacity=0.2)
viewer.add_labels(holes.astype(np.uint8), name="holes")
viewer.add_image(hole_distance.astype(np.float32), name="hole_distance")

<Image layer 'hole_distance' at 0x18c7b0f3e50>

In [21]:
def analyse_internal_pores_per_slice(
    mask,
    voxel_size_um=(2.0, 2.0, 2.0),
    min_pore_area_um2=16.0,
    max_pore_area_fraction_of_slice=0.15,
    use_gpu_edt=True,
):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    z_um, y_um, x_um = voxel_size_um
    pixel_area_um2 = y_um * x_um

    pores_records = []
    slice_records = []
    pore_labels_3d = np.zeros(mask.shape, dtype=np.int32)
    pore_radius_map_3d = np.zeros(mask.shape, dtype=np.float32)
    global_pore_id = 1
    total_pore_area_um2 = 0.0
    total_filled_area_um2 = 0.0

    for z in range(mask.shape[0]):
        vessel_slice = mask[z].astype(bool)
        filled_slice = ndi.binary_fill_holes(vessel_slice)
        internal_pores = filled_slice & ~vessel_slice

        filled_area_um2 = float(np.count_nonzero(filled_slice)) * pixel_area_um2
        total_filled_area_um2 += filled_area_um2

        slice_area_um2 = float(vessel_slice.size) * pixel_area_um2
        max_pore_area_um2 = float(max_pore_area_fraction_of_slice) * slice_area_um2

        if not np.any(internal_pores):
            slice_records.append({
                'z_slice': z,
                'n_internal_pores': 0,
                'pore_area_fraction_in_filled_vascular_area': 0.0,
                'median_pore_area_um2': np.nan,
                'p90_pore_area_um2': np.nan,
                'median_max_inscribed_radius_um': np.nan
            })
            continue

        labeled, n_labels = ndi.label(internal_pores, structure=np.ones((3, 3), dtype=np.uint8))
        if n_labels == 0:
            slice_records.append({
                'z_slice': z,
                'n_internal_pores': 0,
                'pore_area_fraction_in_filled_vascular_area': 0.0,
                'median_pore_area_um2': np.nan,
                'p90_pore_area_um2': np.nan,
                'median_max_inscribed_radius_um': np.nan
            })
            continue

        if use_gpu_edt:
            dist_map_um = cp.asnumpy(
                ndi_gpu.distance_transform_edt(
                    cp.asarray(internal_pores, dtype=cp.uint8),
                    sampling=(float(y_um), float(x_um)),
                )
            )
        else:
            dist_map_um = edt(internal_pores, sampling=(y_um, x_um))

        labels_flat = labeled.ravel()
        area_counts = np.bincount(labels_flat, minlength=n_labels + 1)[1:].astype(np.float64)
        label_ids = np.arange(1, n_labels + 1, dtype=np.int32)
        area_um2_all = area_counts * pixel_area_um2
        valid_mask = (area_um2_all >= min_pore_area_um2) & (area_um2_all <= max_pore_area_um2)
        valid_label_ids = label_ids[valid_mask]

        if valid_label_ids.size == 0:
            slice_records.append({
                'z_slice': z,
                'n_internal_pores': 0,
                'pore_area_fraction_in_filled_vascular_area': 0.0,
                'median_pore_area_um2': np.nan,
                'p90_pore_area_um2': np.nan,
                'median_max_inscribed_radius_um': np.nan
            })
            continue

        max_radius_um_all = np.asarray(
            ndi.maximum(dist_map_um, labels=labeled, index=valid_label_ids),
            dtype=float,
        )
        centroids = ndi.center_of_mass(
            np.ones_like(labeled, dtype=np.float32),
            labels=labeled,
            index=valid_label_ids,
        )
        centroids = np.asarray(centroids, dtype=float)
        if centroids.ndim == 1:
            centroids = centroids[None, :]

        areas_valid_um2 = area_um2_all[valid_mask]
        n_valid = valid_label_ids.size

        global_ids = np.arange(global_pore_id, global_pore_id + n_valid, dtype=np.int32)
        id_map = np.zeros(n_labels + 1, dtype=np.int32)
        id_map[valid_label_ids] = global_ids
        pore_labels_3d[z] = id_map[labeled]

        radius_map = np.zeros(n_labels + 1, dtype=np.float32)
        radius_map[valid_label_ids] = max_radius_um_all.astype(np.float32)
        pore_radius_map_3d[z] = radius_map[labeled]

        for i in range(n_valid):
            area_um2 = float(areas_valid_um2[i])
            max_radius_um = float(max_radius_um_all[i])
            centroid_y_um = float(centroids[i, 0]) * y_um
            centroid_x_um = float(centroids[i, 1]) * x_um

            pores_records.append({
                'pore_id': int(global_ids[i]),
                'z_slice': z,
                'area_um2': area_um2,
                'equivalent_radius_um': float(np.sqrt(area_um2 / np.pi)),
                'max_inscribed_radius_um': max_radius_um,
                'centroid_z_um': float(z) * z_um,
                'centroid_y_um': centroid_y_um,
                'centroid_x_um': centroid_x_um
            })

        slice_total_pore_area_um2 = float(np.sum(areas_valid_um2))
        total_pore_area_um2 += slice_total_pore_area_um2
        global_pore_id += n_valid

        slice_records.append({
            'z_slice': z,
            'n_internal_pores': int(n_valid),
            'pore_area_fraction_in_filled_vascular_area': float(slice_total_pore_area_um2 / max(filled_area_um2, 1e-12)),
            'median_pore_area_um2': float(np.median(areas_valid_um2)),
            'p90_pore_area_um2': float(np.percentile(areas_valid_um2, 90)),
            'median_max_inscribed_radius_um': float(np.median(max_radius_um_all))
        })

    pores_df = pd.DataFrame(pores_records)
    pore_slice_stats_df = pd.DataFrame(slice_records)

    if pores_df.empty:
        pore_summary_df = pd.DataFrame([{
            'total_internal_pore_count': 0,
            'internal_pore_area_fraction_in_filled_vascular_area': 0.0,
            'median_internal_pore_area_um2': np.nan,
            'p90_internal_pore_area_um2': np.nan,
            'median_internal_pore_max_inscribed_radius_um': np.nan,
            'p90_internal_pore_max_inscribed_radius_um': np.nan
        }])
    else:
        pore_summary_df = pd.DataFrame([{
            'total_internal_pore_count': int(len(pores_df)),
            'internal_pore_area_fraction_in_filled_vascular_area': float(total_pore_area_um2 / max(total_filled_area_um2, 1e-12)),
            'median_internal_pore_area_um2': float(np.median(pores_df['area_um2'])),
            'p90_internal_pore_area_um2': float(np.percentile(pores_df['area_um2'], 90)),
            'median_internal_pore_max_inscribed_radius_um': float(np.median(pores_df['max_inscribed_radius_um'])),
            'p90_internal_pore_max_inscribed_radius_um': float(np.percentile(pores_df['max_inscribed_radius_um'], 90))
        }])

    return pores_df, pore_slice_stats_df, pore_summary_df, pore_labels_3d, pore_radius_map_3d

In [ ]:
selected_compact_metric_cols = [
    'vessel_volume_fraction',
    'vessel_length_per_chip_volume_um_inverse2',
    'sprouts_per_chip_volume_um_inverse3',
    'junctions_per_chip_volume_um_inverse3',
    'median_sprout_and_branch_tortuosity',
    'median_cs_area_over_characteristic_area',
    'median_junction_dist_nearest_junction_per_characteristic_length',
    'median_sprout_dist_nearest_endpoint_per_characteristic_length',
    'fractal_dimension',
    'lacunarity',
    'internal_pore_area_fraction_in_filled_vascular_area',
    'total_internal_pore_density_per_vessel_volume_um_inverse3',
    'median_internal_pore_area_um2',
    'p90_internal_pore_area_um2',
    'median_internal_pore_max_inscribed_radius_um',
]

display(global_metrics_df[selected_compact_metric_cols])

,total_internal_pore_count,internal_pore_area_fraction_in_filled_vascular_area,median_internal_pore_area_um2,p90_internal_pore_area_um2,median_internal_pore_max_inscribed_radius_um
0,14273,0.260219,2056.0,18602.4,18.439089
